# 🎯 Module 06: Regularization Techniques

> **Stage:** Deep Learning  
> **Module:** Regularization Techniques  
> **Framework:** PyTorch  
> **Date:** 06 June 2026

---

In this notebook, we move beyond *just building* neural networks and focus on making them **generalize**.

We will cover:

- Overfitting vs underfitting
- Bias and variance
- L1 / L2 regularization
- Weight decay & AdamW
- Dropout
- Early stopping
- Data augmentation
- BatchNorm & LayerNorm
- `train()` vs `eval()`

Everything will be hands-on with **PyTorch**.

## 📚 Learning Objectives

By the end of this notebook, you should be able to:

- Explain the difference between overfitting and underfitting
- Understand bias and variance trade-offs
- Apply L1 / L2 regularization and weight decay
- Use Dropout correctly in PyTorch
- Implement early stopping from scratch
- Apply data augmentation for image data
- Use BatchNorm and LayerNorm appropriately
- Correctly switch between `train()` and `eval()` modes
- Diagnose overfitting from loss curves
- Choose the right regularization technique for a given problem

---
## ⚙️ 0. Setup & Imports

Let's import the libraries we'll need throughout this notebook.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 🧠 1. What Is Regularization?

**Regularization** refers to any technique that helps a model **generalize better** rather than simply memorize the training data.

```text
Training data
      ↓
Learn useful patterns
      ↓
Generalize to unseen data
```

Without regularization, a large neural network can **memorize** the training examples, achieving near-perfect training accuracy but poor test accuracy.

### The Real Goal of ML

> Learn patterns that work on **new** data — not just the training set.

---
## 📉 2. Overfitting vs Underfitting

### Overfitting

The model performs **very well on training data** but **poorly on unseen data**.

```text
Training accuracy    = 99%
Validation accuracy  = 75%
```

**Signs:**
```text
Training loss   ↓↓↓
Validation loss ↓ then ↑
```

### Underfitting

The model **cannot learn the underlying pattern** sufficiently.

```text
Training accuracy    = 65%
Validation accuracy  = 63%
```

**Causes:**
- Model too simple
- Insufficient training
- Poor features / data
- Excessive regularization

---
## 📊 3. Generalization, Bias & Variance

The true objective of machine learning:

> Learn patterns that **transfer** to new data.

### High Bias
```text
Model too simple
↓
Underfitting
```

### High Variance
```text
Model too sensitive to training data
↓
Overfitting
```

Regularization generally tries to **reduce excessive variance**.

⚠️ **But note:** More regularization is *not* always better. Too much causes **underfitting**.

---
## 🧮 4. L1 Regularization

Adds a penalty based on the **absolute values** of weights:

$$L_{total} = L_{data} + \lambda \sum_i |w_i|$$

- $L_{data}$ = original loss
- $\lambda$ = regularization strength
- $w_i$ = model weights

L1 encourages **sparse weights** (some weights go to exactly zero).

---
## 🧮 5. L2 Regularization

Adds a **squared-weight penalty**:

$$L_{total} = L_{data} + \lambda \sum_i w_i^2$$

Large weights receive a stronger penalty.

```text
Large weights
      ↓
Higher penalty
      ↓
Encourage smaller weights
```

### L1 vs L2

| Feature | L1 | L2 |
| --- | --- | --- |
| Penalty | $\|w\|$ | $w^2$ |
| Encourages zeros | More strongly | Less directly |
| Effect | Sparse weights | Smaller weights |
| Neural-net usage | Less common | **Very common** |

---
## ⚖️ 6. Weight Decay

**Weight decay** is closely related to L2 regularization:

$$w \leftarrow w - \eta \left(\nabla L + \lambda w\right)$$

Modern optimizers implement weight decay differently. This distinction matters for **Adam-based optimizers**.

---
## 🔧 7. Adam vs AdamW

With **Adam**, regularization and adaptive optimization interact in a *coupled* way.

**AdamW** uses **decoupled weight decay** — usually the better choice when combining Adam-style optimization with weight decay.

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.01
)
```

In [ ]:
# Quick demo: Adam vs AdamW setup
demo_model = nn.Linear(10, 1)

adam_opt  = optim.Adam(demo_model.parameters(),  lr=1e-3, weight_decay=0.01)
adamw_opt = optim.AdamW(demo_model.parameters(), lr=1e-3, weight_decay=0.01)

print('Adam optimizer:  ', adam_opt)
print('AdamW optimizer: ', adamw_opt)
print('\nBoth use weight_decay=0.01, but AdamW decouples it from adaptive scaling.')

---
## 🎲 8. Dropout

Dropout randomly **removes some activations** during training.

```text
Before:  [1.2, 0.7, 2.1, 0.4]
Dropout: [1.2, 0,   2.1, 0  ]
```

The network cannot rely too heavily on specific neurons — this encourages **robust representations**.

### Why It Works

```text
Without dropout:  Neuron A → very important
With dropout:     A may disappear
                  ↓
                  Network learns multiple useful representations
```

In [ ]:
# Demo: see dropout in action
torch.manual_seed(0)
x = torch.ones(1, 8) * 1.5
dropout = nn.Dropout(p=0.3)

dropout.train()
print('Training mode (dropout active):')
for _ in range(3):
    print('  ', dropout(x).numpy())

dropout.eval()
print('\nEval mode (dropout disabled):')
print('  ', dropout(x).numpy())

### PyTorch Dropout in a Model

```python
model = nn.Sequential(
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(64, 10)
)
```

`p=0.3` → ~30% of activations are dropped during training.

---
## 🔀 9. `train()` vs `eval()` — Critical!

Dropout, BatchNorm, and other layers behave **differently** depending on mode.

| Mode | Dropout | BatchNorm |
| --- | --- | --- |
| `model.train()` | Active | Uses batch stats |
| `model.eval()`  | Disabled | Uses running stats |

### Correct Inference Pattern

```python
model.eval()

with torch.no_grad():
    output = model(x)
```

⚠️ Forgetting `model.eval()` before evaluation is a **very common bug** — it produces inconsistent and overly noisy predictions.

---
## ⏹️ 10. Early Stopping

Stop training when **validation performance stops improving**.

```text
Epoch    Train Loss    Val Loss
1        0.80          0.85
2        0.60          0.65
3        0.45          0.50
4        0.32          0.42
5        0.25          0.40   ← best val
6        0.18          0.44   ← val increases
7        0.12          0.51   ← overfitting!
```

### Patience

Instead of stopping immediately, allow training to continue for a few epochs without improvement:

```text
patience = 3

Validation improves → continue
No improvement      → wait
Still no improvement → stop
```

In [ ]:
# A reusable EarlyStopping utility
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = None
        self.counter = 0
        self.stop = False
        self.best_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

print('EarlyStopping class ready.')

---
## 🖼️ 11. Data Augmentation

Data augmentation creates modified versions of training examples:

```text
Original image
     ↓
Flip / Rotate / Crop / Resize / Color changes
     ↓
Additional training examples
```

The goal is to expose the model to **reasonable variations**.

⚠️ Do **not** apply transformations that change the meaning of the label (e.g., flipping digits 6↔9).

In [ ]:
# Example torchvision augmentation pipeline for images
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

print('Train augmentation pipeline:')
print(train_transform)

# Regularization vs Data Augmentation:
#   Weight regularization → constrains the MODEL
#   Data augmentation    → varies the DATA

---
## 🧱 12. Batch Normalization

Normalizes activations using **statistics computed from the current mini-batch**:

```text
Layer output
   ↓
Normalize
   ↓
Scale + Shift
   ↓
Next layer
```

PyTorch: `nn.BatchNorm1d(64)` or `nn.BatchNorm2d(64)`.

### Train vs Eval for BatchNorm

| Phase | Behavior |
| --- | --- |
| Training | Uses batch statistics |
| Evaluation | Uses stored running statistics |

---
## 🧱 13. Layer Normalization

Normalizes across **features within an individual example** — no batch dependency.

PyTorch: `nn.LayerNorm(512)`. Common in **Transformers**.

### BatchNorm vs LayerNorm

| Feature | BatchNorm | LayerNorm |
| --- | --- | --- |
| Depends on batch stats | Yes | No |
| Common in CNNs | Yes | Less common |
| Common in Transformers | Less common | **Yes** |
| Works with tiny batches | Can be problematic | Generally better |

In [ ]:
# Demo: BatchNorm vs LayerNorm tensor shapes
x_bn = torch.randn(4, 8)   # batch=4, features=8

bn = nn.BatchNorm1d(8)
ln = nn.LayerNorm(8)

bn.train(); ln.train()
y_bn = bn(x_bn)
y_ln = ln(x_bn)

print('Input shape:', x_bn.shape)
print('BatchNorm mean per-feature (should be ~0):', y_bn.mean(dim=0).detach().numpy().round(3))
print('LayerNorm mean per-sample  (should be ~0):', y_ln.mean(dim=1).detach().numpy().round(3))

---
## 🧪 14. Practical Experiment

We'll build a **synthetic classification dataset** that is easy to overfit, then train 3 models:

- **Model A:** No regularization
- **Model B:** Dropout
- **Model C:** Weight decay (AdamW)

We'll compare **generalization gaps**.

$$\text{Generalization Gap} = \text{Train Accuracy} - \text{Val Accuracy}$$

In [ ]:
# Create a small, noisy, high-dimensional dataset that encourages overfitting
torch.manual_seed(42)

N_TRAIN = 1000
N_VAL   = 500
D_IN    = 100
N_CLASS = 2

def make_data(n):
    X = torch.randn(n, D_IN)
    # Non-linear, noisy label: only first 5 features matter
    logits = 3.0 * X[:, 0] - 2.0 * X[:, 1] + 1.5 * X[:, 2] - X[:, 3] + X[:, 4]
    y = (logits + 0.5 * torch.randn(n) > 0).long()
    return X, y

X_train, y_train = make_data(N_TRAIN)
X_val,   y_val   = make_data(N_VAL)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=128, shuffle=False)

print('Train:', X_train.shape, 'Val:', X_val.shape)
print('Positive class fraction (train):', y_train.float().mean().item())

In [ ]:
# Model factory
def build_model(dropout_p=0.0):
    layers = [
        nn.Linear(D_IN, 256), nn.ReLU(),
        nn.Linear(256, 256),  nn.ReLU(),
        nn.Linear(256, 128),  nn.ReLU(),
        nn.Linear(128, N_CLASS)
    ]
    if dropout_p > 0:
        # Insert dropout after each ReLU
        layers = [
            nn.Linear(D_IN, 256), nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(256, 256),  nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(256, 128),  nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(128, N_CLASS)
        ]
    return nn.Sequential(*layers).to(device)

print('Model factory ready.')

In [ ]:
# Training / evaluation helpers
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return total_loss / total, total_correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return total_loss / total, total_correct / total

print('Train/eval helpers ready.')

In [ ]:
# Full training loop for a given configuration
def run_experiment(name, dropout_p=0.0, weight_decay=0.0, epochs=40, lr=1e-3):
    torch.manual_seed(42)
    model = build_model(dropout_p=dropout_p)
    criterion = nn.CrossEntropyLoss()

    # AdamW decouples weight decay from the adaptive update
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)

    gap = history['train_acc'][-1] - history['val_acc'][-1]
    print(f'{name:22s} | train_acc={history["train_acc"][-1]:.3f}  '
          f'val_acc={history["val_acc"][-1]:.3f}  gap={gap:+.3f}')
    return history

# Run the three experiments
hist_A = run_experiment('A: No regularization')
hist_B = run_experiment('B: Dropout (p=0.5)', dropout_p=0.5)
hist_C = run_experiment('C: Weight decay (1e-2)', weight_decay=1e-2)

In [ ]:
# Plot the loss curves for all three experiments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for hist, label in [(hist_A, 'A: None'), (hist_B, 'B: Dropout'), (hist_C, 'C: Weight decay')]:
    axes[0].plot(hist['train_loss'], linestyle='--', label=f'{label} (train)')
    axes[0].plot(hist['val_loss'],   linestyle='-',  label=f'{label} (val)')
    axes[1].plot(hist['train_acc'], linestyle='--', label=f'{label} (train)')
    axes[1].plot(hist['val_acc'],   linestyle='-',  label=f'{label} (val)')

axes[0].set_title('Loss curves');        axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[1].set_title('Accuracy curves');    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
axes[0].grid(alpha=0.3);    axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 🔍 What to Look For

- **Model A (no reg):** training loss keeps dropping, but validation loss may start rising → classic **overfitting**.
- **Model B (dropout):** training accuracy lower, but val accuracy closer → **smaller generalization gap**.
- **Model C (weight decay):** smoother curves, larger weights discouraged → often better generalization.

> Remember: the best model is not the one with the lowest **training** loss — it's the one with the best **validation** performance.

---
## 🧭 15. Choosing the Right Regularization

### If the model **overfits**:

```text
More training data
Data augmentation
Weight decay
Dropout
Early stopping
Smaller model
```

### If the model **underfits**:

```text
Less regularization
Larger model
Better features
Longer training
```

⚠️ Do **not** automatically add every regularization technique. Each one is a tool for a specific problem.

---
## 🚨 16. Common Mistakes

### ❌ Mistake 1 — Forgetting `model.eval()` during inference
Dropout and BatchNorm behave differently at inference. Always call `model.eval()`.

### ❌ Mistake 2 — Too much dropout
`Dropout = 0.8` can make learning unnecessarily difficult. Typical range: **0.1 – 0.5**.

### ❌ Mistake 3 — Excessive weight decay
Very strong regularization prevents the model from learning useful patterns.

### ❌ Mistake 4 — Data leakage
Augmentation / preprocessing must respect the **train / val / test** separation. Never use info from validation/test data to train the model.

---
## 🎤 17. Interview Questions

1. What is overfitting?
2. What is underfitting?
3. What is regularization?
4. Why does regularization help?
5. What is L1 regularization?
6. What is L2 regularization?
7. What is weight decay?
8. What is dropout?
9. Why is dropout disabled during inference?
10. What is early stopping?
11. What is data augmentation?
12. What is BatchNorm?
13. What is LayerNorm?
14. BatchNorm vs LayerNorm — when would you use which?
15. What happens if regularization is too strong?
16. Why use `model.eval()`?
17. Why can validation loss increase while training loss decreases?

---
## 🏋️ 18. Hands-On Exercises

Try these on your own. Solutions are open-ended!

### Exercise 1 — Add BatchNorm
Modify `build_model` to insert `nn.BatchNorm1d` after each `Linear` (before ReLU).
Re-run the three experiments and compare.

### Exercise 2 — Early Stopping
Use the `EarlyStopping` class you defined above. Set `patience=5` and stop when validation loss stops improving.
Which epoch does training stop for Model A?

### Exercise 3 — Sweep Dropout
Try `dropout_p ∈ {0.0, 0.1, 0.3, 0.5, 0.7}` and plot final validation accuracy vs dropout rate.
Where does performance peak?

### Exercise 4 — LayerNorm Variant
Replace BatchNorm with `nn.LayerNorm` and observe the difference on this tabular task.

### Exercise 5 — Data Augmentation (vision)
Load `torchvision.datasets.MNIST` with and without augmentation. Train the same CNN on both and compare validation accuracy.

### Exercise 6 — L1 vs L2
Implement L1 regularization manually by adding `lambda * sum(p.abs().sum() for p in model.parameters())` to the loss. Compare with L2 / weight decay.

---
## ✅ 19. Knowledge Check

You should now be able to:

- [ ] Explain overfitting
- [ ] Explain underfitting
- [ ] Explain generalization
- [ ] Explain bias and variance
- [ ] Explain L1
- [ ] Explain L2
- [ ] Explain weight decay
- [ ] Explain AdamW
- [ ] Explain dropout
- [ ] Use `Dropout` in PyTorch
- [ ] Explain early stopping
- [ ] Explain data augmentation
- [ ] Explain BatchNorm
- [ ] Explain LayerNorm
- [ ] Explain `train()` vs `eval()`
- [ ] Identify overfitting from training/validation curves
- [ ] Choose an appropriate regularization technique

---
## 🏁 20. Key Takeaways

The core problem:

```text
Model memorizes training data
          ↓
Poor generalization
          ↓
Overfitting
```

Regularization provides different tools to combat this:

```text
L1 / L2           →  Control weights
Dropout           →  Reduce neuron dependency
Data Augmentation →  Increase useful variation
Early Stopping    →  Stop before excessive fitting
BatchNorm/LayerNorm → Improve training behavior
```

### 🧭 The Golden Rule

> **Regularization should improve generalization, not simply make training loss smaller.**